# WRDS Universe Fetch
**Run Cell 1 first** (enter credentials), then **run Cells 2–5 in order**.

In [15]:
# ── Cell 1: Login ─────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
import pandas as pd
import sqlalchemy as sa
import wrds

PROJECT   = Path(os.getenv('ATC_PROJECT_ROOT', Path('..').resolve())).resolve()
UNIV_DIR  = PROJECT / 'data' / 'universe'
UNIV_DIR.mkdir(parents=True, exist_ok=True)
SP_CACHE   = UNIV_DIR / 'sp_constituents_wrds.parquet'
RU3K_CACHE = UNIV_DIR / 'ru3k_constituents_crsp.parquet'
LINK_CACHE = UNIV_DIR / 'crsp_compustat_link.parquet'

print('Enter your WRDS credentials below:')
db = wrds.Connection()
print('Connected!')

def qry(sql, date_cols=None):
    """Run SQL against open db connection, avoiding the raw_sql() immutabledict bug."""
    df = pd.read_sql_query(sa.text(sql), db.connection)
    if date_cols:
        for c in date_cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c])
    return df

Enter your WRDS credentials below:
WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connected!


In [16]:
# ── Cell 2: S&P 500 / 400 / 600 membership ────────────────────────────────────
# Auto-detect gvkeyx codes by constituent count: SP500≈500, SP400≈400, SP600≈600.
# This works regardless of what names WRDS assigns internally.

print('Discovering S&P index GVKEYs...')
idx_counts = qry("""
    SELECT gvkeyx,
           COUNT(DISTINCT gvkey) AS n_distinct_members
    FROM comp.idxcst_his
    GROUP BY gvkeyx
    ORDER BY n_distinct_members DESC
""")
print(idx_counts.to_string())

# Map by constituent count: closest to 500/400/600 (allowing historical additions)
def closest_to(df, target, col='n_distinct_members', exclude=()):
    mask = ~df['gvkeyx'].isin(exclude)
    sub = df[mask].copy()
    sub['dist'] = (sub[col] - target).abs()
    return sub.loc[sub['dist'].idxmin(), 'gvkeyx']

gvkey_sp500 = closest_to(idx_counts, 500)
gvkey_sp400 = closest_to(idx_counts, 400, exclude=[gvkey_sp500])
gvkey_sp600 = closest_to(idx_counts, 600, exclude=[gvkey_sp500, gvkey_sp400])

print(f'\nAuto-detected: SP500={gvkey_sp500}, SP400={gvkey_sp400}, SP600={gvkey_sp600}')

print('\nFetching membership intervals...')
sp_hist = qry(f"""
    SELECT
        gvkey::bigint  AS gvkey,
        CASE gvkeyx
            WHEN '{gvkey_sp500}' THEN 'SP500'
            WHEN '{gvkey_sp400}' THEN 'SP400'
            WHEN '{gvkey_sp600}' THEN 'SP600'
        END            AS idx_type,
        "from"::date   AS start_dt,
        COALESCE(thru::date, '2035-12-31'::date) AS end_dt
    FROM comp.idxcst_his
    WHERE gvkeyx IN ('{gvkey_sp500}', '{gvkey_sp400}', '{gvkey_sp600}')
    ORDER BY gvkey, start_dt
""", date_cols=['start_dt', 'end_dt'])

sp_hist = sp_hist[sp_hist['idx_type'].notna()].copy()
sp_hist.to_parquet(SP_CACHE, index=False)
print(f'\nSaved {len(sp_hist):,} rows → {SP_CACHE}')
print(sp_hist['idx_type'].value_counts())

Discovering S&P index GVKEYs...
     gvkeyx  n_distinct_members
0    031855                1500
1    165188                1196
2    000010                1176
3    165186                 725
4    150918                 647
5    030824                 600
6    151015                 514
7    000003                 500
8    165181                 458
9    165157                 436
10   000208                 428
11   024248                 400
12   150927                 347
13   165174                 340
14   165171                 302
15   150912                 263
16   129023                 262
17   128878                 255
18   165170                 242
19   153376                 201
20   128938                 193
21   129041                 191
22   150913                 178
23   129019                 168
24   129378                 161
25   150916                 149
26   165155                 143
27   118341                 141
28   129040                 121
29   129

In [17]:
# ── Cell 3: CRSP monthly market cap (~3–5 min) ────────────────────────────────
print('Fetching CRSP monthly market cap...')
crsp_me = qry("""
    SELECT msf.permno,
           msf.date,
           ABS(msf.prc) * msf.shrout AS me
    FROM crsp.msf AS msf
    JOIN crsp.msenames AS names
      ON msf.permno = names.permno
     AND msf.date BETWEEN names.namedt AND names.nameendt
    WHERE msf.date >= '2009-01-01'
      AND names.shrcd  IN (10, 11)
      AND names.exchcd IN (1, 2, 3)
      AND ABS(msf.prc) > 1.0
      AND msf.shrout  > 0
    ORDER BY msf.date, msf.permno
""", date_cols=['date'])
print(f'  {len(crsp_me):,} rows fetched')

Fetching CRSP monthly market cap...
  702,064 rows fetched


In [18]:
# ── Cell 4: PERMNO → GVKEY link, then close connection ────────────────────────
print('Fetching PERMNO → GVKEY link...')
crsp_link = qry("""
    SELECT
        lpermno::bigint AS permno,
        gvkey::bigint   AS gvkey,
        linkdt::date                                  AS link_start,
        COALESCE(linkenddt::date, '2035-12-31'::date) AS link_end
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LC', 'LU', 'LS')
      AND linkprim IN ('P', 'J', 'C')
    ORDER BY permno, link_start
""", date_cols=['link_start', 'link_end'])
crsp_link.to_parquet(LINK_CACHE, index=False)
print(f'  Saved {len(crsp_link):,} rows → {LINK_CACHE}')
db.close()
print('WRDS connection closed.')

Fetching PERMNO → GVKEY link...
  Saved 40,934 rows → /Users/chaithanyapakala/Downloads/NLP_FinalProject/data/universe/crsp_compustat_link.parquet
WRDS connection closed.


In [19]:
# ── Cell 5: Build Russell 3000 PIT membership from CRSP market cap ────────────
print('Building Russell 3000 PIT membership (top 3000 by market cap at each June recon)...')

def last_friday_june(year):
    d = pd.Timestamp(f'{year}-06-30')
    return d - pd.Timedelta(days=(d.weekday() - 4) % 7)

recon_dates = [last_friday_june(y) for y in range(2009, pd.Timestamp.now().year + 2)]
crsp_me['month'] = crsp_me['date'].dt.to_period('M')

ru3k_records = []
for i, recon in enumerate(recon_dates):
    recon_month = recon.to_period('M')
    month_me    = crsp_me[crsp_me['month'] == recon_month]
    if month_me.empty:
        month_me = crsp_me[crsp_me['month'] == (recon_month - 1)]
    if month_me.empty:
        print(f'  Warning: no data near {recon.date()}, skipping')
        continue

    top3k = set(month_me.nlargest(3000, 'me')['permno'])
    link_at = crsp_link[
        (crsp_link['link_start'] <= recon) & (crsp_link['link_end'] >= recon)
    ][['permno', 'gvkey']].drop_duplicates('permno')
    mapped = link_at[link_at['permno'].isin(top3k)]

    end_date = (recon_dates[i + 1] - pd.Timedelta(days=1)
                if i + 1 < len(recon_dates) else pd.Timestamp('2035-12-31'))

    for _, row in mapped.iterrows():
        ru3k_records.append({'gvkey': int(row['gvkey']),
                             'start_dt': recon, 'end_dt': end_date})

    print(f'  {recon.date()}: {len(mapped):,} GVKEYs')

ru3k_hist = pd.DataFrame(ru3k_records)
ru3k_hist.to_parquet(RU3K_CACHE, index=False)
print(f'\nSaved {len(ru3k_hist):,} rows → {RU3K_CACHE}')
print(f'Unique GVKEYs ever in Russell 3000: {ru3k_hist["gvkey"].nunique():,}')
print('\n✓ All done!')

Building Russell 3000 PIT membership (top 3000 by market cap at each June recon)...
  2009-06-26: 2,996 GVKEYs
  2010-06-25: 2,996 GVKEYs
  2011-06-24: 2,993 GVKEYs
  2012-06-29: 2,997 GVKEYs
  2013-06-28: 2,996 GVKEYs
  2014-06-27: 2,996 GVKEYs
  2015-06-26: 2,992 GVKEYs
  2016-06-24: 2,993 GVKEYs
  2017-06-30: 2,996 GVKEYs
  2018-06-29: 2,994 GVKEYs
  2019-06-28: 2,995 GVKEYs
  2020-06-26: 2,996 GVKEYs
  2021-06-25: 2,982 GVKEYs
  2022-06-24: 2,990 GVKEYs
  2023-06-30: 2,994 GVKEYs
  2024-06-28: 2,995 GVKEYs

Saved 47,901 rows → /Users/chaithanyapakala/Downloads/NLP_FinalProject/data/universe/ru3k_constituents_crsp.parquet
Unique GVKEYs ever in Russell 3000: 6,626

✓ All done!
